# Trovare i collegamenti tra tabelle SQL

Questo notebook legge tutti i file `.sql` di una cartella, estrae tabelle e colonne e cerca i **possibili collegamenti** (chiavi esterne mancanti) in base a:

1. **colonne con lo stesso nome** in più tabelle
2. **colonne con nomi simili** (`COD_ART` / `CODART` / `CodArticolo`)
3. **nomi che rimandano a una tabella** (`id_articolo` → tabella `articoli`) e chiavi primarie

Alla fine salva tutto in un file Excel e un file `.sql` con gli `ALTER TABLE` proposti, da controllare a mano.

**Come usarlo:** modifica la cella *Configurazione*, poi esegui tutte le celle in ordine (menu *Run → Run All Cells*).

Pacchetti necessari: si installano con la cella *0. Installazione* qui sotto (`sqlglot` è facoltativo ma consigliato).

## 0. Installazione (solo la prima volta)

Installa le librerie necessarie. Va eseguita una sola volta: poi si può saltare.

Se compare un errore come *"Nessuna connessione selezionata"*, il notebook è aperto con il kernel sbagliato: in alto a destra scegli il kernel **Python 3** (non SQL).

In [ ]:
!pip install pandas openpyxl sqlglot pyvis ipython

## 1. Configurazione

In [ ]:
from pathlib import Path

# Cartella con i file .sql (di default i file di prova in "esempi").
# Per il tuo database, ad esempio: Path(r"C:\Users\TUO_UTENTE\Desktop\sql_files")
cartella = Path("esempi")

# Dialetto SQL: "mysql", "tsql" (SQL Server), "oracle", "postgres", ...
DIALETTO = "mysql"

# Colonne troppo generiche per essere collegamenti: compaiono ovunque.
# Aggiungi qui quelle che nel risultato ti sembrano "rumore".
GENERICHE = {
    "id", "nome", "descrizione", "descr", "note", "data", "stato", "tipo",
    "codice", "valore", "importo", "created_at", "updated_at", "deleted_at",
    "data_ins", "data_mod", "utente_ins", "utente_mod", "attivo", "flag",
}

# Soglia di somiglianza tra nomi (da 0 a 1): più alta = meno risultati ma più precisi
SOGLIA_SIMILI = 0.85

print("Cartella esiste:", cartella.exists())
print("File .sql trovati:", len(list(cartella.glob("*.sql"))))

## 2. Funzioni di supporto

In [ ]:
import re
import logging
from difflib import SequenceMatcher
from itertools import combinations

import pandas as pd
from IPython.display import display

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 120)

try:
    import sqlglot
    from sqlglot import exp
    logging.getLogger("sqlglot").setLevel(logging.ERROR)   # niente warning "falling back..."
    HA_SQLGLOT = True
except ImportError:
    HA_SQLGLOT = False
    print("sqlglot non installato: uso solo il parser interno (pip install sqlglot)")

RE_CREATE = re.compile(r"CREATE\s+(?:TEMPORARY\s+)?TABLE\b[^(;]*?\(", re.I | re.S)
VINCOLI = ("PRIMARY", "KEY", "UNIQUE", "INDEX", "CONSTRAINT", "FOREIGN",
           "FULLTEXT", "SPATIAL", "CHECK", "PERIOD")


def togli_commenti(sql):
    sql = re.sub(r"/\*.*?\*/", " ", sql, flags=re.S)     # /* ... */ e /*! ... */
    sql = re.sub(r"--[^\n]*", " ", sql)                   # -- commento
    if DIALETTO == "mysql":
        sql = re.sub(r"#[^\n]*", " ", sql)                # # commento (solo MySQL)
    return sql


def pulisci(nome):
    return nome.strip().strip('`"[]').strip()


def nome_tabella(intestazione):
    nome = re.sub(r"^\s*CREATE\s+(?:TEMPORARY\s+)?TABLE\s+(?:IF\s+NOT\s+EXISTS\s+)?",
                  "", intestazione, flags=re.I).strip()
    return pulisci(nome.split(".")[-1])


def chiusura_parentesi(sql, start):
    """Posizione della ')' che chiude la '(' in start, ignorando le stringhe."""
    depth, quote = 0, None
    for i in range(start, len(sql)):
        c = sql[i]
        if quote:
            if c == quote:
                quote = None
        elif c in "'\"`":
            quote = c
        elif c == "(":
            depth += 1
        elif c == ")":
            depth -= 1
            if depth == 0:
                return i
    return None


def split_virgole(body):
    """Divide il corpo del CREATE TABLE sulle virgole di primo livello."""
    parti, buf, depth, quote = [], "", 0, None
    for c in body:
        if quote:
            if c == quote:
                quote = None
        elif c in "'\"`":
            quote = c
        elif c == "(":
            depth += 1
        elif c == ")":
            depth -= 1
        elif c == "," and depth == 0:
            parti.append(buf.strip())
            buf = ""
            continue
        buf += c
    if buf.strip():
        parti.append(buf.strip())
    return parti


def e_vincolo(riga):
    if riga[:1] in ("`", '"', "["):        # nome tra virgolette: è una colonna
        return False
    return riga.split()[0].upper() in VINCOLI


def colonne_regex(body):
    """Parser di riserva: estrae (nome, tipo) da ogni riga del CREATE TABLE."""
    risultato = []
    for riga in split_virgole(body):
        if not riga or e_vincolo(riga):
            continue
        m = re.match(r'([`"\[][^`"\]]+[`"\]]|[^\s,]+)\s+\[?(\w+)\]?(\s*\([^)]*\))?', riga)
        if m:
            tipo = (m.group(2) + (m.group(3) or "")).replace(" ", "").upper()
            risultato.append((pulisci(m.group(1)), tipo))
    return risultato


def pk_da_body(body):
    """Chiave primaria dichiarata nel CREATE TABLE (lista di colonne)."""
    for riga in split_virgole(body):
        if re.match(r"(CONSTRAINT\s+\S+\s+)?PRIMARY\s+KEY", riga, re.I):
            m = re.search(r"\(([^)]*)\)", riga)
            if m:
                return [pulisci(c.split("(")[0]) for c in m.group(1).split(",")]
        elif re.search(r"\bPRIMARY\s+KEY\b", riga, re.I) and not e_vincolo(riga):
            return [pulisci(riga.split()[0])]
    return []


def normalizza(nome):
    """COD_ART, CodArt, cod-art -> codart"""
    return re.sub(r"[^a-z0-9]", "", str(nome).lower())


def tipo_base(t):
    if not isinstance(t, str) or not t:
        return ""
    m = re.match(r"\w+", t)
    return m.group(0).upper() if m else ""


def radice(nome):
    """articoli / articolo / tbl_articoli / items -> stessa radice"""
    n = re.sub(r"^(tbl|tab|tb|t)_", "", str(nome).lower())
    if len(n) > 3:
        n = re.sub(r"s$", "", n)
    if len(n) > 3:
        n = re.sub(r"[aeio]$", "", n)
    return n.replace("_", "")


def riferimento(col):
    """id_articolo / articolo_id / articoloId / cod_articolo -> 'articolo'"""
    c = re.sub(r"([a-z])([A-Z])", r"\1_\2", str(col)).lower()
    m = (re.fullmatch(r"(?:id|cod|codice|fk|num)_?(.+)", c)
         or re.fullmatch(r"(.+?)_?(?:id|cod|codice|fk)", c))
    return m.group(1) if m else None

## 3. Lettura dei file ed estrazione di tabelle e colonne

In [ ]:
righe, pk, errori = [], {}, []

for file in sorted(cartella.glob("*.sql")):
    sql = togli_commenti(file.read_text(encoding="utf-8", errors="replace"))

    for m in RE_CREATE.finditer(sql):
        apertura = m.end() - 1
        chiusura = chiusura_parentesi(sql, apertura)
        if chiusura is None:
            errori.append({"file": file.name, "inizio": sql[m.start():m.start() + 80]})
            continue

        tabella = nome_tabella(sql[m.start():apertura])
        body = sql[apertura + 1:chiusura]

        # 1) sqlglot sul solo "CREATE TABLE nome ( ... )", senza ENGINE/COMMENT/ecc.
        colonne, metodo = None, "regex"
        if HA_SQLGLOT:
            try:
                stmt = sqlglot.parse_one(sql[m.start():chiusura + 1], read=DIALETTO)
                if isinstance(stmt, exp.Create):
                    defs = list(stmt.find_all(exp.ColumnDef))
                    if defs:
                        colonne = [(c.name, c.kind.sql(dialect=DIALETTO).upper() if c.kind else None)
                                   for c in defs]
                        metodo = "sqlglot"
            except Exception:
                pass

        # 2) parser di riserva
        if colonne is None:
            colonne = colonne_regex(body)

        for nome, tipo in colonne:
            righe.append({"file": file.name, "tabella": tabella,
                          "colonna": nome, "tipo": tipo, "metodo": metodo})

        pk[tabella] = pk_da_body(body) or pk.get(tabella, [])

df_schema = pd.DataFrame(righe, columns=["file", "tabella", "colonna", "tipo", "metodo"])
col = df_schema.drop_duplicates(["tabella", "colonna"]).copy()

print("Tabelle trovate:", col["tabella"].nunique())
print("Colonne trovate:", len(col))
print(df_schema["metodo"].value_counts().to_string())
if errori:
    print("\nCREATE TABLE non letti (parentesi non chiusa):")
    display(pd.DataFrame(errori))
display(df_schema)

## 4. Chiavi primarie

Usa quelle dichiarate nei file; se una tabella non ne ha, prova a indovinarla (`id`, `id_tabella`, `tabella_id`).

In [ ]:
col_per_tab = col.groupby("tabella")["colonna"].apply(list).to_dict()
pk_indovinata = {}

for t, colonne_t in col_per_tab.items():
    if not pk.get(t):
        for c in colonne_t:
            rif = riferimento(c)
            if c.lower() == "id" or (rif and radice(rif) == radice(t)):
                pk[t] = [c]
                pk_indovinata[t] = True
                break

riepilogo = pd.DataFrame([
    {"tabella": t,
     "n_colonne": len(colonne_t),
     "chiave primaria": ", ".join(pk.get(t, [])) or "-",
     "pk dichiarata": bool(pk.get(t)) and not pk_indovinata.get(t, False)}
    for t, colonne_t in sorted(col_per_tab.items())
])

print("Tabelle senza chiave primaria:", (riepilogo["chiave primaria"] == "-").sum())
display(riepilogo)

## 5. Colonne con lo stesso nome in più tabelle

L'indizio più forte. La colonna **"è PK in"** dice in quale tabella quella colonna è chiave primaria: di solito è la tabella "principale" a cui puntano le altre. **"parte di PK composta in"** dice in quali tabelle la colonna fa parte di una chiave primaria su più colonne.

In [ ]:
col["norm"] = col["colonna"].map(normalizza)
col["tipo_base"] = col["tipo"].map(tipo_base)
generiche_norm = {normalizza(g) for g in GENERICHE}
col_utili = col[~col["norm"].isin(generiche_norm)]

# in quali tabelle ogni nome (normalizzato) è chiave primaria
pk_singola = {t: c[0] for t, c in pk.items() if len(c) == 1}
pk_composta = {t: c for t, c in pk.items() if len(c) > 1}
pk_di = {}
for t, c in pk_singola.items():
    pk_di.setdefault(normalizza(c), []).append(t)
parte_pk_di = {}                                   # colonne che fanno parte di una PK composta
for t, colonne_pk in pk_composta.items():
    for c in colonne_pk:
        parte_pk_di.setdefault(normalizza(c), []).append(t)

stesso_nome = (
    col_utili.groupby("norm")
        .agg(colonna=("colonna", "first"),
             n_tabelle=("tabella", "nunique"),
             tabelle=("tabella", lambda x: ", ".join(sorted(set(x)))),
             tipi=("tipo", lambda x: ", ".join(sorted(set(map(str, x))))))
        .query("n_tabelle > 1")
        .reset_index()
)
n_tipi = col_utili.groupby("norm")["tipo_base"].nunique()
stesso_nome["tipi coerenti"] = stesso_nome["norm"].map(n_tipi) == 1
stesso_nome["è PK in"] = stesso_nome["norm"].map(lambda n: ", ".join(pk_di.get(n, [])) or "-")
stesso_nome["parte di PK composta in"] = stesso_nome["norm"].map(lambda n: ", ".join(parte_pk_di.get(n, [])) or "-")
stesso_nome = (stesso_nome.drop(columns="norm")
                          .sort_values("n_tabelle", ascending=False)
                          .reset_index(drop=True))

print("Colonne con lo stesso nome in più tabelle:", len(stesso_nome))
display(stesso_nome)

## 6. Colonne con nomi simili (non identici)

Ci saranno falsi positivi: tieni soprattutto le coppie con **stesso tipo = True**. Se i risultati sono troppi, alza `SOGLIA_SIMILI`.

In [ ]:
gruppi = col_utili.groupby("norm")
uniche = list(gruppi.groups)
simili = []

for a, b in combinations(uniche, 2):
    if abs(len(a) - len(b)) > 4:
        continue
    contiene = (a in b or b in a) and min(len(a), len(b)) >= 4
    sm = SequenceMatcher(None, a, b)
    if not contiene and (sm.real_quick_ratio() < SOGLIA_SIMILI or sm.quick_ratio() < SOGLIA_SIMILI):
        continue                       # scarto veloce
    score = sm.ratio()
    if score < SOGLIA_SIMILI and not contiene:
        continue
    for ra in gruppi.get_group(a).itertuples():
        for rb in gruppi.get_group(b).itertuples():
            if ra.tabella != rb.tabella:
                simili.append({
                    "tabella 1": ra.tabella, "colonna 1": ra.colonna, "tipo 1": ra.tipo,
                    "tabella 2": rb.tabella, "colonna 2": rb.colonna, "tipo 2": rb.tipo,
                    "somiglianza": round(score, 2),
                    "stesso tipo": ra.tipo_base == rb.tipo_base,
                })

df_simili = pd.DataFrame(simili)
if not df_simili.empty:
    df_simili = (df_simili.sort_values(["stesso tipo", "somiglianza"], ascending=False)
                          .reset_index(drop=True))

print("Coppie di colonne con nomi simili:", len(df_simili))
display(df_simili)

## 7. Chiavi esterne candidate

Per ogni colonna cerca la tabella a cui potrebbe puntare:

- **alta**: la colonna ha lo stesso nome della chiave primaria di un'altra tabella
- **media**: il nome rimanda al nome di una tabella (`id_articolo` → `articoli`)
- **alta (PK composta)**: la tabella contiene **tutte** le colonne della chiave primaria composta di un'altra tabella (`id_ordine` + `num_riga` → `ordini_righe`). Il vincolo proposto è su più colonne insieme.

Se due tabelle hanno la stessa chiave primaria composta, il collegamento viene proposto in entrambe le direzioni: tieni quello che va verso la tabella principale.

**tipi ok = False** significa che il vincolo fallirebbe: prima va sistemato il tipo.

In [ ]:
per_radice = {}
for t in col_per_tab:
    per_radice.setdefault(radice(t), []).append(t)

tipi = {(r.tabella, r.colonna): r.tipo for r in col.itertuples()}
cand = {}

# Regola C: la tabella contiene tutte le colonne di una chiave primaria composta
colonne_norm = {t: {} for t in col_per_tab}       # tabella -> {nome normalizzato: nome vero}
for r in col.itertuples():
    colonne_norm[r.tabella].setdefault(r.norm, r.colonna)
composte = {}
for dest, colonne_pk in pk_composta.items():
    norm_pk = [normalizza(c) for c in colonne_pk]
    if all(n in generiche_norm for n in norm_pk):
        continue
    for t, nomi in colonne_norm.items():
        if t != dest and all(n in nomi for n in norm_pk):
            composte[(t, dest)] = ([nomi[n] for n in norm_pk],
                                   [colonne_norm[dest].get(n, c) for n, c in zip(norm_pk, colonne_pk)])

for r in col.itertuples():
    t, c = r.tabella, r.colonna
    if pk.get(t) == [c]:
        continue                                   # è la PK della sua tabella

    # Regola A: stesso nome della PK di un'altra tabella
    if r.norm != "id":
        for dest in pk_di.get(r.norm, []):
            if dest != t:
                cand[(t, c, dest)] = ("alta", pk_singola[dest])

    # Regola B: il nome della colonna rimanda a una tabella
    rif = riferimento(c)
    if rif:
        for dest in per_radice.get(radice(rif), []):
            if dest != t and (t, c, dest) not in cand and (t, dest) not in composte:
                cand[(t, c, dest)] = ("media", pk_singola.get(dest, "?"))

righe_fk = []
for (t, c, dest), (conf, col_dest) in cand.items():
    tipo_o, tipo_d = tipi.get((t, c)), tipi.get((dest, col_dest))
    righe_fk.append({
        "tabella": t, "colonna": c, "tipo": tipo_o,
        "→ tabella": dest, "→ colonna": col_dest, "tipo dest": tipo_d,
        "affidabilità": conf,
        "tipi ok": col_dest != "?" and tipo_base(tipo_o) == tipo_base(tipo_d),
        "sql": (f"ALTER TABLE {t} ADD CONSTRAINT fk_{t}_{c} "
                f"FOREIGN KEY ({c}) REFERENCES {dest} ({col_dest});"),
    })

for (t, dest), (col_o, col_d) in composte.items():
    tipi_o = [tipi.get((t, c)) for c in col_o]
    tipi_d = [tipi.get((dest, c)) for c in col_d]
    righe_fk.append({
        "tabella": t, "colonna": ", ".join(col_o), "tipo": ", ".join(map(str, tipi_o)),
        "→ tabella": dest, "→ colonna": ", ".join(col_d), "tipo dest": ", ".join(map(str, tipi_d)),
        "affidabilità": "alta",
        "tipi ok": all(tipo_base(a) == tipo_base(b) for a, b in zip(tipi_o, tipi_d)),
        "sql": (f"ALTER TABLE {t} ADD CONSTRAINT fk_{t}_{dest} "
                f"FOREIGN KEY ({', '.join(col_o)}) REFERENCES {dest} ({', '.join(col_d)});"),
    })

df_fk = pd.DataFrame(righe_fk, columns=["tabella", "colonna", "tipo", "→ tabella", "→ colonna",
                                        "tipo dest", "affidabilità", "tipi ok", "sql"])
df_fk = df_fk.sort_values(["affidabilità", "tabella", "colonna"]).reset_index(drop=True)

# Colonne con nome "da chiave" che non trovano nessuna tabella: da guardare a mano
gia_trovate = {(t, c.strip()) for t, cs in zip(df_fk["tabella"], df_fk["colonna"]) for c in cs.split(",")}
sospette = col[
    col["colonna"].map(lambda x: riferimento(x) is not None)
    & ~col.apply(lambda r: (r.tabella, r.colonna) in gia_trovate, axis=1)
    & ~col.apply(lambda r: pk.get(r.tabella) == [r.colonna], axis=1)
][["tabella", "colonna", "tipo"]].reset_index(drop=True)

print("FK candidate:", len(df_fk), "| di cui affidabilità alta:", (df_fk["affidabilità"] == "alta").sum(),
      "| su PK composte:", len(composte))
display(df_fk)
print("Colonne che sembrano chiavi ma senza tabella corrispondente:", len(sospette))
display(sospette)

## 8. Tabelle isolate

Tabelle che non compaiono in nessun collegamento trovato: sono quelle su cui indagare a mano.

In [ ]:
collegate = set(df_fk["tabella"]) | set(df_fk["→ tabella"])
for _, r in stesso_nome.iterrows():
    collegate |= set(r["tabelle"].split(", "))
if not df_simili.empty:
    collegate |= set(df_simili["tabella 1"]) | set(df_simili["tabella 2"])

isolate = sorted(set(col_per_tab) - collegate)
print("Tabelle senza nessun collegamento trovato:", len(isolate))
for t in isolate:
    print(f"  {t}: {', '.join(col_per_tab[t])}")

## 9. Salvataggio

Crea nella cartella dei file `.sql`:

- `possibili_collegamenti.xlsx` con un foglio per ogni analisi
- `fk_candidate.sql` con gli `ALTER TABLE` proposti (**da controllare prima di eseguirli**)

In [ ]:
file_excel = cartella / "possibili_collegamenti.xlsx"
with pd.ExcelWriter(file_excel) as xls:
    riepilogo.to_excel(xls, sheet_name="Tabelle", index=False)
    df_schema.to_excel(xls, sheet_name="Colonne", index=False)
    stesso_nome.to_excel(xls, sheet_name="Stesso nome", index=False)
    df_simili.to_excel(xls, sheet_name="Nomi simili", index=False)
    df_fk.to_excel(xls, sheet_name="FK candidate", index=False)
    sospette.to_excel(xls, sheet_name="Da verificare", index=False)
    pd.DataFrame({"tabella": isolate}).to_excel(xls, sheet_name="Isolate", index=False)

file_alter = cartella / "fk_candidate.sql"
with open(file_alter, "w", encoding="utf-8") as f:
    f.write("-- Chiavi esterne CANDIDATE: controllarle prima di eseguirle\n")
    for conf in ("alta", "media"):
        f.write(f"\n-- ===== Affidabilità {conf} =====\n")
        for _, r in df_fk[df_fk["affidabilità"] == conf].iterrows():
            nota = "" if r["tipi ok"] else "   -- ATTENZIONE: tipi diversi o colonna destinazione sconosciuta"
            f.write(r["sql"] + nota + "\n")

print("Salvato:", file_excel)
print("Salvato:", file_alter)

## 10. Schema grafico dei collegamenti

Crea `schema_collegamenti.html` e lo apre nel browser. Ogni casella è una tabella (passa il mouse per vedere le colonne, 🔑 = chiave primaria); le frecce verdi sono collegamenti ad affidabilità alta, quelle arancioni tratteggiate media. Le tabelle grigie sono isolate. Puoi trascinare le caselle e fare zoom.

Serve `pip install pyvis`.

In [ ]:
# pip install pyvis
import webbrowser
from pyvis.network import Network

net = Network(height="850px", width="100%", directed=True, cdn_resources="in_line")
net.barnes_hut(gravity=-8000, spring_length=200)

# Una casella per tabella: passando il mouse si vedono le colonne
for t, colonne_t in col_per_tab.items():
    elenco = "\n".join(
        f"{'🔑 ' if c in pk.get(t, []) else ''}{c}  {tipi.get((t, c)) or ''}"
        for c in colonne_t
    )
    net.add_node(t, label=t, title=f"{t}\n\n{elenco}", shape="box",
                 color="#e0e0e0" if t in isolate else "#97c2fc")

# Una freccia per ogni FK candidata: verde = alta, arancione tratteggiata = media
colori = {"alta": "#2e7d32", "media": "#ef6c00"}
for _, r in df_fk.iterrows():
    net.add_edge(r["tabella"], r["→ tabella"],
                 label=r["colonna"],
                 title=f'{r["tabella"]}.{r["colonna"]} → {r["→ tabella"]}.{r["→ colonna"]}'
                       f'\naffidabilità: {r["affidabilità"]}'
                       + ("" if r["tipi ok"] else "\nATTENZIONE: tipi diversi"),
                 color=colori[r["affidabilità"]],
                 dashes=r["affidabilità"] == "media")

file_html = cartella / "schema_collegamenti.html"
net.write_html(str(file_html))
print("Salvato:", file_html)
webbrowser.open(file_html.resolve().as_uri())